In [ ]:
import sempy.fabric as fabric
import pandas as pd

# Function to clean up DataFrame column names
def clean_column_names(df):
    return df.rename(columns=lambda x: x.strip("[]"))

# Get workspaces and filter on dedicated capacity
df_workspaces = fabric.list_workspaces()
df_workspaces = df_workspaces.rename(columns={
    'Is Read Only': 'Is_Read_Only',
    'Id': 'ID',
    'Is On Dedicated Capacity': 'Is_On_Dedicated_Capacity',
    'Default Dataset Storage Format': 'Default_Dataset_Storage_Format',
    'Capacity Id': 'Capacity_ID'
})
workspace_ids = df_workspaces.query('Is_On_Dedicated_Capacity == True')['ID'].tolist()

# List all objects and filter to 'SemanticModel' (datasets)
df = pd.concat([fabric.list_items(workspace=ws) for ws in workspace_ids], ignore_index=True)
df = df.rename(columns={'Display Name': 'Display_Name', 'Workspace Id': 'Workspace_ID'})
df_datasets = df[df['Type'] == 'SemanticModel']

# Merge datasets with workspaces for easier access
df_datasets = pd.merge(
    df_datasets,
    df_workspaces.rename(columns={'ID': 'Workspace_ID'}),
    how='left',
    on='Workspace_ID'
)

# Generic function to evaluate DAX and process results
def evaluate_dax(workspace_name, dataset_name, dax_string, results):
    try:
        doc = fabric.evaluate_dax(
            workspace=workspace_name,
            dataset=dataset_name,
            dax_string=dax_string
        )
        df_doc = pd.DataFrame(doc)
        df_doc = clean_column_names(df_doc)
        df_doc["Workspace_Name"] = workspace_name
        df_doc["Dataset_Name"] = dataset_name
        results.append(df_doc)
    except Exception as e:
        print(f"Error processing workspace: {workspace_name}, dataset: {dataset_name}. Error: {e}")

# Define queries for Measures, Tables, Columns, Relationships
queries = {
    "Measures": """
        EVALUATE
        SELECTCOLUMNS(
            INFO.VIEW.MEASURES(),
            "Name",[Name],
            "Table",[Table],
            "DataType",[DataType],
            "Description",[Description],
            "IsHidden",[IsHidden],
            "Expression",[Expression],
            "DisplayFolder", [DisplayFolder],
            "FormatString",[FormatString],
            "KPIID",[KPIID],
            "State",[State],
            "FormatStringDefinition",[FormatStringDefinition],
            "IsSimpleMeasure",[IsSimpleMeasure],
            "DataCategory",[DataCategory],
            "DetailRowsDefinition",[DetailRowsDefinition],
            "LineageTag",[LineageTag]
        )
    """,
    "Tables": """
        EVALUATE
        SELECTCOLUMNS(
            INFO.VIEW.TABLES(),
            "Name",[Name],
            "Model",[Model],
            "DataCategory",[DataCategory],
            "Description",[Description],
            "IsHidden",[IsHidden],
            "StorageMode",[StorageMode],
            "TableStorage",[TableStorage],
            "Expression",[Expression],
            "ShowAsVariationOnly",[ShowAsVariationOnly],
            "IsPrivate",[IsPrivate],
            "CalculationGroupPrecedence",[CalculationGroupPrecedence],
            "LineageTag", [LineageTag]
        )
    """,
    "Columns": """
        EVALUATE
        SELECTCOLUMNS(
            INFO.VIEW.COLUMNS(),
            "Name",[Name],
            "Table",[Table],
            "DataType",[DataType],
            "DataCategory",[DataCategory],
            "Description",[Description],
            "IsHidden",[IsHidden],
            "IsUnique",[IsUnique],
            "IsKey",[IsKey],
            "IsNullable",[IsNullable],
            "Alignment",[Alignment],
            "Summarizeby",[Summarizeby],
            "ColumnStorage",[ColumnStorage],
            "Type", [Type],
            "SourceColumn",[SourceColumn],
            "Expression",[Expression],
            "FormatString",[FormatString],
            "IsAvailableInMDX",[IsAvailableInMDX],
            "SortByColumn",[SortByColumn],
            "GroupingBehaviour",[GroupingBehavior],
            "SourceProviderType",[SourceProviderType],
            "DisplayFolder",[DisplayFolder],
            "AlternateOf",[AlternateOf],
            "LineageTag",[LineageTag]
        )
    """,
    "Relationships": """
        EVALUATE
        SELECTCOLUMNS(
            INFO.VIEW.RELATIONSHIPS(),
            "Name",[Name],
            "Relationship",[Relationship],
            "Model",[Model],
            "IsActive",[IsActive],
            "CrossFilteringBehavior",[CrossFilteringBehavior],
            "RelyOnReferentialIntegrity",[RelyOnReferentialIntegrity],
            "FromTable",[FromTable],
            "FromColumn",[FromColumn],
            "FromCardinality",[FromCardinality],
            "ToTable",[ToTable],
            "ToColumn",[ToColumn],
            "ToCardinality",[ToCardinality],
            "State", [State],
            "SecurityFilteringBehavior",[SecurityFilteringBehavior]
        )
    """
}

# Process each query
def process_query(query_name, dax_string):
    results = []
    for _, row in df_datasets.iterrows():
        evaluate_dax(row["Name"], row["Display_Name"], dax_string, results)
    if results:
        combined_df = pd.concat(results, ignore_index=True)
        spark_df = spark.createDataFrame(combined_df)
        spark_df.write.format("delta").mode("overwrite").save(f"Tables/documentation_{query_name}")
    else:
        print(f"No results found for {query_name}.")

# Run all queries
for query_name, dax_string in queries.items():
    process_query(query_name, dax_string)